# Chapter 5 — Sweep parameter points

TARGET API · CONVERGING · not executable on the current runtime

> **TARGET API / CONVERGING — not executable on the current runtime.**
> The target parameter sweep API is shown without numerical execution or
> assumed outputs.

A `ParameterSet` selects one immutable point. A `ParameterSpace` selects
an ordered collection of points for one already-bound Plan, one View,
and one request. This Chapter rebuilds the resonator so the six-point
sweep is complete in a clean kernel.

## Lesson 5.1 — Bind the same physical inputs

In [ ]:
from scnsim import (
    CircuitPlan,
    ParameterDefinitions,
    ParameterSpec,
    components,
    units as u,
)

inputs = ParameterDefinitions(id="readout_design")
capacitance = inputs.parameter(
    id="capacitance", baseline=110.0 * u.fF, spec=ParameterSpec(unit=u.fF)
)
inductance = inputs.parameter(
    id="inductance", baseline=5.8 * u.nH, spec=ParameterSpec(unit=u.nH)
)
plan = CircuitPlan(id="primitive_resonator")
resonator = plan.subsystem(id="resonator")
capacitor = resonator.add(
    components.capacitor(id="capacitor", capacitance=capacitance)
)
inductor = resonator.add(
    components.inductor(id="inductor", inductance=inductance)
)
resonator_bus = resonator.bus(id="terminal")
resonator.parallel(
    id="parallel_lc",
    start=resonator_bus,
    branches=((capacitor,), (inductor,)),
    end=resonator.ground,
)
terminal = resonator.expose_pin(id="terminal", at=resonator_bus)

In [ ]:
signal_bus = plan.bus(id="signal_boundary")
resonator_root_bus = plan.bus(id="resonator_node")
coupling_cap = plan.add(
    components.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
plan.series(
    id="coupling",
    start=signal_bus,
    elements=(coupling_cap,),
    end=resonator_root_bus,
)
plan.link(id="resonator_terminal", endpoints=(resonator_root_bus, terminal))
plan.add_port(
    id="signal_in",
    at=signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
resonator_node = resonator_root_bus.node

The two refs are already physically consumed. The following spaces
therefore select values for this Plan; they do not add inputs or alter
topology.

## Lesson 5.2 — Choose a Cartesian grid or listed points

`grid` forms the exact author-ordered 3 × 2 Cartesian product:
capacitance is the first axis and inductance the second. `points`
instead preserves the two explicit paired samples in the listed order;
it is not a Cartesian product.

In [ ]:
from scnsim import ParameterSet, ParameterSpace

grid = ParameterSpace.grid(
    axes={
        capacitance: [100.0, 110.0, 120.0] * u.fF,
        inductance: [5.8, 6.0] * u.nH,
    },
    fixed=ParameterSet(),
)
listed_points = ParameterSpace.points(
    (
        ParameterSet({capacitance: 100.0 * u.fF, inductance: 5.8 * u.nH}),
        ParameterSet({capacitance: 120.0 * u.fF, inductance: 6.0 * u.nH}),
    )
)

Axes and `fixed` must be disjoint. A scalar ref cannot receive an array
as one value. Repeated listed positions remain repeated; no
nearest-point or interpolation rule is implied.

## Lesson 5.3 — Request, select, and collect an existing quantity

In [ ]:
from scnsim import CircuitRun, DiagonalRootSpec, ReductionPipeline

run = CircuitRun(plan=plan, workspace="workspaces/parameter-course")
quantity_view = run.original.reduce(
    ReductionPipeline().retain(resonator_node)
)
root_spec = DiagonalRootSpec(
    coordinate=resonator_node,
    root_hint=6.0 * u.GHz,
)
sweep = run.evaluate(quantity_view, root_spec, parameters=grid)

For a grid, `sweep.points[2, 0]` identifies the third capacitance and
first inductance sample. Listed spaces use ordinal indices. Each point
retains its complete effective `ParameterSet`, success status, and
either a result or its typed failure; one numerical failure never
becomes a zero or fabricated result.

In [ ]:
from IPython.display import display

sweep.show()

In [ ]:
grid_point = sweep.points[2, 0]
if grid_point.succeeded:
    display(grid_point.result.frequency)
else:
    display(grid_point.failure)

In [ ]:
selected = sweep.select(parameters=ParameterSet({capacitance: 110.0 * u.fF}))

In [ ]:
selected.show()

In [ ]:
line = selected.collect(quantity=root_spec.frequency)
line.show(x=inductance)

In [ ]:
frequency_grid = sweep.collect(quantity=root_spec.frequency)
frequency_grid.show(x=capacitance, y=inductance)

In [ ]:
listed_sweep = run.evaluate(quantity_view, root_spec, parameters=listed_points)
listed_point = listed_sweep.points[1]
if listed_point.succeeded:
    display(listed_point.result.frequency)
else:
    display(listed_point.failure)

`select` treats supplied `ParameterSet` entries as exact filters and
omitted entries as wildcards, preserving the source indices and
duplicate points. `collect` reads only the already-requested typed
`root_spec.frequency` selector, retaining axes, units, and failure mask;
it never launches a calculation. A listed collection remains scattered
points rather than silently becoming a heatmap.

[Previous](04_define_parameters.qmd) · [Next](06_optimize_primitive.qmd)